# Review Participation

Stratification of the cohort along its *discussion rate*, defined per repository as the share of merged pull requests that received at least one comment from a non-bot account other than the author before merge. The empirical distribution motivates the stratum cuts; two repositories are then drawn per stratum under a fixed seed.

In [1]:
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import polars as pl
from plotly.subplots import make_subplots

from benchmark_notebook import theme  # noqa: F401  registers thesis template
from benchmark_notebook.io import CloudStorage

In [2]:
store = CloudStorage(bucket="bug-request-bucket")

In [3]:
cohort = pl.from_arrow(
    store.asset(
        key="scope-repository-cohort",
        version="6.0.0",
        location="main",
        partition="default",
    )
).select("name")
cohort.height

65

## Per-PR review state

The issue catalog already filters reviews to those submitted before merge and is restricted to merged pull requests, so per-PR aggregation reduces to a boolean over the `reviews` list. Bot reviews are excluded to keep the signal at the level of human review involvement.

In [4]:
catalog = (
    pl.from_arrow(
        store.asset(
            key="scope-issue-catalog",
            version="4.0.0",
            location="main",
            partition="default",
        )
    )
    .filter(pl.col("type") == "change")
    .select(
        "name",
        "author",
        "number",
        "closed",
        pl.col("payload").struct.field("reviews").alias("reviews"),
    )
)

catalog.height

925617

In [5]:
prs = catalog.with_columns(
    pl.col("reviews")
    .list.eval(
        (pl.element().struct.field("type") == "user")
        & (pl.element().struct.field("state") == "CHANGES_REQUESTED")
    )
    .list.any()
    .fill_null(False)
    .alias("contested")
).select("name", "number", "author", "closed", "contested")

participation = (
    prs.group_by("name")
    .agg(
        pl.len().alias("merged_prs"),
        pl.col("contested").sum().alias("contested_prs"),
    )
    .with_columns((pl.col("contested_prs") / pl.col("merged_prs")).alias("share"))
    .join(cohort, on="name", how="inner")
    .sort("share")
)

participation

name,merged_prs,contested_prs,share
str,u32,u32,f64
"""netboxlabs/diode-netbox-plugin""",139,0,0.0
"""NVIDIA-NeMo/Run""",322,1,0.003106
"""Energinet-DataHub/opengeh-pyth…",209,1,0.004785
"""cohere-ai/cohere-compass-sdk""",165,1,0.006061
"""cds-snc/sre-bot""",613,4,0.006525
…,…,…,…
"""SwissDataScienceCenter/renku-d…",709,209,0.294781
"""fivetran/fivetran_connector_sd…",382,125,0.327225
"""fox-it/flow.record""",152,69,0.453947


## Distribution

The empirical histogram across the cohort is reported with a placeholder bin count. The bin definition for stratified sampling is deliberately deferred until the distribution shape has been inspected.

In [6]:
bins = 20
lo, hi = 0.0, 1.0

(
    go.Figure()
    .add_trace(
        go.Histogram(
            x=participation["share"],
            marker=dict(
                color=pio.templates["thesis"].layout.colorway[3],
                line=dict(width=0),
            ),
            xbins=dict(start=lo, end=hi + (hi - lo) / bins, size=(hi - lo) / bins),
            autobinx=False,
        )
    )
    .update_xaxes(title_text="Review Participation", range=[0, 1])
    .update_yaxes(title_text="Repositories")
    .update_layout(
        width=800,
        height=400,
        showlegend=False,
        bargap=0.02,
        margin=dict(l=70, r=30, t=40, b=60),
    )
    .show(config={"toImageButtonOptions": {"format": "svg"}})
)

## Discussion rate

A second participation signal, closer to McIntosh's *discussion rate*: the share of merged pull requests that received at least one comment from a non-bot account. Comments are drawn from both GitHub channels — top-level conversation comments and inline review-thread comments — and tagged by author type (`bot`/`user`) and source (`conversation`/`review`) at collection time, so the bot filter is applied here at analysis. The denominator is merged pull requests authored by non-bots, mirroring the cohort definition.

In [7]:
discussion = pl.from_arrow(
    store.asset(
        key="scope-change-discussion",
        version="1.0.0",
        location="main",
        partition="default",
    )
)

discussion

name,number,comments
str,i64,list[struct[4]]
"""mbta/dmap-import""",129,[]
"""mbta/dmap-import""",128,[]
"""mbta/dmap-import""",127,[]
"""mbta/dmap-import""",126,[]
"""mbta/dmap-import""",125,[]
…,…,…
"""ansible/django-ansible-base""",5,"[{""sonarqubecloud"",""bot"",""conversation"",""2023-10-25T20:11:51Z""}]"
"""ansible/django-ansible-base""",4,"[{""sonarqubecloud"",""bot"",""conversation"",""2023-10-25T19:19:05Z""}]"
"""ansible/django-ansible-base""",3,"[{""sonarqubecloud"",""bot"",""conversation"",""2023-10-25T18:52:42Z""}, {""github-advanced-security"",""bot"",""conversation"",""2023-10-25T18:52:44Z""}]"


In [8]:
discussed = (
    discussion.join(
        catalog.select("name", "number", "author", "closed").rename(
            {"author": "pr_author"}
        ),
        on=["name", "number"],
        how="inner",
    )
    .explode("comments")
    .unnest("comments")
    .filter(
        (pl.col("type") == "user")
        & (pl.col("author") != pl.col("pr_author"))
        & (pl.col("created") <= pl.col("closed"))
    )
    .group_by(["name", "number"])
    .agg(pl.len().alias("n_user_comments"))
    .with_columns((pl.col("n_user_comments") > 0).alias("discussed"))
    .join(
        catalog.select("name", "number"),
        on=["name", "number"],
        how="right",
    )
    .with_columns(pl.col("discussed").fill_null(False))
    .select("name", "number", "discussed")
)

discussion_rate = (
    discussed.group_by("name")
    .agg(
        pl.len().alias("merged_prs"),
        pl.col("discussed").sum().alias("discussed_prs"),
    )
    .with_columns((pl.col("discussed_prs") / pl.col("merged_prs")).alias("share"))
    .join(cohort, on="name", how="inner")
    .sort("share")
)

discussion_rate


name,merged_prs,discussed_prs,share
str,u32,u32,f64
"""cds-snc/sre-bot""",613,68,0.11093
"""opentdf/tests""",265,40,0.150943
"""netboxlabs/diode-netbox-plugin""",139,26,0.18705
"""stytchauth/stytch-python""",241,48,0.19917
"""aws-deadline/deadline-cloud-te…",166,36,0.216867
…,…,…,…
"""fox-it/dissect.target""",995,680,0.683417
"""optuna/optuna""",3763,2684,0.713261
"""equinor/everest-models""",124,93,0.75


In [9]:
bins = 20
lo, hi = 0.0, 1.0

(
    go.Figure()
    .add_trace(
        go.Histogram(
            x=discussion_rate["share"],
            marker=dict(
                color=pio.templates["thesis"].layout.colorway[3],
                line=dict(width=0),
            ),
            xbins=dict(start=lo, end=hi + (hi - lo) / bins, size=(hi - lo) / bins),
            autobinx=False,
        )
    )
    .update_xaxes(title_text="Discussion Rate", range=[0, 1])
    .update_yaxes(title_text="Repositories")
    .update_layout(
        width=800,
        height=400,
        showlegend=False,
        bargap=0.02,
        margin=dict(l=70, r=30, t=40, b=60),
    )
    .show()
)

## Stratified sampling

The cohort is split into three discussion-rate strata at 0.32 and 0.48, and two repositories are drawn from each without replacement. The draw uses a fixed seed over a NumPy generator, with the strata processed in the order low, mid, high, so the selection is reproducible.

In [10]:
seed, k, cuts = 451600, 2, [0.32, 0.48]

base = discussion_rate.with_columns(
    pl.col("share").cut(breaks=cuts, labels=["low", "mid", "high"]).alias("stratum")
)
strata = {
    s: base.filter(pl.col("stratum") == s)["name"].to_list()
    for s in ["low", "mid", "high"]
}

rng = np.random.default_rng(seed)
picked = []
for names in strata.values():
    indices = rng.choice(len(names), size=k, replace=False)
    picked.extend(names[i] for i in indices)

sample = base.filter(pl.col("name").is_in(picked)).select("name", "stratum", "share")
sample

name,stratum,share
str,cat,f64
"""OpenJobDescription/openjd-mode…","""low""",0.240876
"""mistralai/mistral-common""","""low""",0.306569
"""cohere-ai/cohere-compass-sdk""","""mid""",0.327273
"""microsoft/Olive""","""mid""",0.459476
"""kedro-org/kedro-plugins""","""high""",0.508885
"""awslabs/graphstorm""","""high""",0.653802


## Marginal distributions over the outline

Descriptive statistics over per-PR text length, token count, and changed-file scale characterise the sample prior to corrective-filter construction. Per-repository PR counts surface the bin-size marginal; per-PR length and file-count distributions surface the outline marginal.

In [11]:
outline = pl.from_arrow(
    store.asset(
        key="scope-change-outline",
        version="3.0.0",
        location="main",
        partition="default",
    )
)

outline = outline.join(sample.select("name"), on="name", how="inner")
outline.height

3731

In [12]:
features = outline.with_columns(
    pl.col("title").fill_null("").str.len_chars().alias("title_chars"),
    pl.col("body").fill_null("").str.len_chars().alias("body_chars"),
    (pl.col("title").fill_null("") + pl.lit("\n") + pl.col("body").fill_null(""))
    .str.split(" ")
    .list.len()
    .alias("approx_tokens"),
    pl.col("files").list.len().alias("file_count"),
    pl.col("files")
    .list.eval(
        pl.element().struct.field("additions") + pl.element().struct.field("deletions")
    )
    .list.sum()
    .alias("loc_changed"),
)

features.select(
    "title_chars", "body_chars", "approx_tokens", "file_count", "loc_changed"
).describe()

statistic,title_chars,body_chars,approx_tokens,file_count,loc_changed
str,f64,f64,f64,f64,f64
"""count""",3731.0,3731.0,3731.0,3731.0,3731.0
"""null_count""",0.0,0.0,0.0,0.0,0.0
"""mean""",45.58617,697.506567,115.931922,7.308228,732.147682
"""std""",19.363304,2368.867338,983.246815,15.844854,17446.054846
"""min""",4.0,0.0,1.0,1.0,0.0
"""25%""",31.0,281.0,45.0,1.0,9.0
"""50%""",44.0,463.0,74.0,3.0,49.0
"""75%""",58.0,680.0,104.0,7.0,231.0
"""max""",142.0,135054.0,59635.0,417.0,751289.0


In [13]:
per_repo = (
    features.group_by("name")
    .agg(
        pl.len().alias("prs"),
        pl.col("body_chars").median().alias("body_chars_median"),
        pl.col("approx_tokens").median().alias("tokens_median"),
        pl.col("file_count").median().alias("file_count_median"),
        pl.col("loc_changed").median().alias("loc_changed_median"),
    )
    .join(sample.select("name", "share"), on="name", how="inner")
    .sort("share")
)

with pl.Config(tbl_rows=-1, tbl_cols=-1, fmt_str_lengths=120):
    print(per_repo)

shape: (6, 7)
┌────────────────┬──────┬───────────────┬───────────────┬───────────────┬───────────────┬──────────┐
│ name           ┆ prs  ┆ body_chars_me ┆ tokens_median ┆ file_count_me ┆ loc_changed_m ┆ share    │
│ ---            ┆ ---  ┆ dian          ┆ ---           ┆ dian          ┆ edian         ┆ ---      │
│ str            ┆ u32  ┆ ---           ┆ f64           ┆ ---           ┆ ---           ┆ f64      │
│                ┆      ┆ f64           ┆               ┆ f64           ┆ f64           ┆          │
╞════════════════╪══════╪═══════════════╪═══════════════╪═══════════════╪═══════════════╪══════════╡
│ OpenJobDescrip ┆ 207  ┆ 1137.0        ┆ 161.0         ┆ 1.0           ┆ 7.0           ┆ 0.240876 │
│ tion/openjd-mo ┆      ┆               ┆               ┆               ┆               ┆          │
│ del-for-python ┆      ┆               ┆               ┆               ┆               ┆          │
│ mistralai/mist ┆ 138  ┆ 71.0          ┆ 16.0          ┆ 2.0           ┆ 19.

In [14]:
panels = [
    {"x": "title_chars", "xaxis": "Title Characters", "log": False},
    {"x": "body_chars", "xaxis": "log10(Body Characters)", "log": True},
    {"x": "file_count", "xaxis": "Changed Files", "log": False},
    {"x": "loc_changed", "xaxis": "log10(Lines Changed)", "log": True},
]

bins = 50
fig = make_subplots(
    rows=2,
    cols=2,
    vertical_spacing=0.16,
    horizontal_spacing=0.12,
)

for i, panel in enumerate(panels):
    row, col = divmod(i, 2)
    row, col = row + 1, col + 1
    series = features[panel["x"]].drop_nulls().cast(pl.Float64)
    if panel["log"]:
        series = series.filter(series > 0).log10()
    series = series.filter(series.is_finite())
    lo, hi = float(series.min()), float(series.max())
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        continue
    step = (hi - lo) / bins

    fig.add_trace(
        go.Histogram(
            x=series,
            marker=dict(
                color=pio.templates["thesis"].layout.colorway[3],
                line=dict(width=0),
            ),
            xbins=dict(start=lo, end=hi + step, size=step),
            autobinx=False,
        ),
        row=row,
        col=col,
    )
    fig.update_xaxes(title_text=panel["xaxis"], row=row, col=col)
    fig.update_yaxes(title_text="PRs", row=row, col=col)

fig.update_layout(
    width=800,
    height=600,
    showlegend=False,
    bargap=0.02,
    margin=dict(l=70, r=30, t=40, b=60),
).show(config={"toImageButtonOptions": {"format": "svg"}})